# Feature Transformation Using Sklearn With ANN

In [1]:
# %pip install pandas

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [3]:
data = pd.read_csv("../01 Datasets/Churn_Modelling.csv")

In [4]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
data.drop(["RowNumber", "CustomerId", "Surname"], axis=1, inplace=True)

# or you can write: data = data.drop([["RowNumber", "CustomerId", "Surname"], axis=1])

In [6]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
# Encoding the categorical variable

label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

In [8]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [9]:
# One hot Encoding
from sklearn.preprocessing import OneHotEncoder

one_hot_geo = OneHotEncoder()
geo_encoder = one_hot_geo.fit_transform(data[["Geography"]])

geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [10]:
one_hot_geo.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [11]:
geo_encoder_df = pd.DataFrame(
    geo_encoder.toarray(), columns=one_hot_geo.get_feature_names_out(["Geography"])
)

In [12]:
geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [13]:
## combine the columns with the original data
data = pd.concat([data.drop("Geography", axis=1), geo_encoder_df], axis=1)

In [14]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [15]:
# Save the encoders and scaler
import pickle

with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)

with open("one_hot_geo.pkl", "wb") as file:
    pickle.dump(one_hot_geo, file)

In [16]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [17]:
X = data.drop("Exited", axis=1)
y = data["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [19]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [20]:
import datetime

In [21]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import warnings

warnings.filterwarnings("ignore")

In [22]:
X_train.shape[1]

12

In [23]:
model = Sequential(
    [
        Dense(64, activation="relu", input_shape=(X_train.shape[1],)),  ## first HL
        Dense(32, activation="relu"),  ## second HL
        Dense(1, activation="sigmoid"),  ## output L
    ]
)

In [24]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [25]:
import tensorflow

opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()

In [26]:
model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])

In [ ]:
## set up the Tensorboard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [28]:
## set up EarlyStopping
early_stopping_callback = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [29]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorflow_callback, early_stopping_callback],
)

Epoch 1/100


250/250 [==============================] - 8s 11ms/step - loss: 0.3964 - accuracy: 0.8353 - val_loss: 0.3557 - val_accuracy: 0.8570
Epoch 2/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3557 - accuracy: 0.8555 - val_loss: 0.3657 - val_accuracy: 0.8545
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3519 - accuracy: 0.8610 - val_loss: 0.3495 - val_accuracy: 0.8540
Epoch 4/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3461 - accuracy: 0.8589 - val_loss: 0.3444 - val_accuracy: 0.8525
Epoch 5/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3428 - accuracy: 0.8608 - val_loss: 0.3490 - val_accuracy: 0.8575
Epoch 6/100
250/250 [==============================] - 1s 5ms/step - loss: 0.3400 - accuracy: 0.8606 - val_loss: 0.3382 - val_accuracy: 0.8545
Epoch 7/100
250/250 [==============================] - 1s 6ms/step - loss: 0.3356 - accuracy: 0.8644 - val_loss: 0.3444 - val_accuracy: 0.8

In [30]:
model.save("model.h5")

In [31]:
%load_ext tensorboard

In [36]:
%tensorboard --logdir logs/fit20260603-144450

Reusing TensorBoard on port 6006 (pid 9544), started 0:00:04 ago. (Use '!kill 9544' to kill it.)

In [37]:
!kill 11776

kill: 11776: No such process
